In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Paths
PROC = Path('../data/processed')
FEAT = Path('../features')
CLASSES = ['glioma', 'meningioma', 'pituitary', 'notumor']

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.11.0+cpu


In [2]:
# Load ResNet50 with ImageNet weights
print("Downloading ResNet50 weights (one-time, ~100 MB)...")
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Strip the final classification layer
# Originally: model.fc maps 2048 -> 1000 (ImageNet classes)
# We replace it with Identity so output is the 2048-D vector before classification
model.fc = nn.Identity()

# Freeze and put in eval mode (no gradient tracking, batch norm stats fixed)
model.eval()
for p in model.parameters():
    p.requires_grad = False

model = model.to(device)
print(f"\nModel ready. Output dimension: 2048")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [01:27<00:00, 1.18MB/s]



Model ready. Output dimension: 2048
Total parameters: 23,508,032


In [3]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # replicate gray to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),  # ImageNet stats
])

# Test on one image
test_img_path = next((PROC / 'Training' / 'glioma').glob('*.png'))
test_img = Image.open(test_img_path).convert('L')
test_tensor = preprocess(test_img).unsqueeze(0).to(device)  # add batch dim

print(f"Input shape: {test_tensor.shape}")  # expect [1, 3, 224, 224]

with torch.no_grad():
    test_feats = model(test_tensor)

print(f"Output shape: {test_feats.shape}")  # expect [1, 2048]
print(f"First 10 features: {test_feats[0, :10].cpu().numpy()}")

Input shape: torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 2048])
First 10 features: [2.3288118e-02 1.0674223e-01 5.3189015e-01 4.8644203e-04 1.1313947e-02
 4.1207392e-03 1.6361152e-01 3.7919182e-02 1.2081215e+00 0.0000000e+00]


In [4]:
def extract_deep_batch(image_paths, batch_size=32):
    """
    Yields (file_paths, features_array) tuples in batches.
    """
    for i in range(0, len(image_paths), batch_size):
        batch = image_paths[i:i+batch_size]
        
        # Load and preprocess all images in batch
        tensors = []
        for p in batch:
            img = Image.open(p).convert('L')
            tensors.append(preprocess(img))
        
        batch_tensor = torch.stack(tensors).to(device)
        
        with torch.no_grad():
            features = model(batch_tensor).cpu().numpy()
        
        yield batch, features


# Quick test on a small batch of 4
test_files = list((PROC / 'Training' / 'glioma').glob('*.png'))[:4]
for paths, feats in extract_deep_batch(test_files, batch_size=4):
    print(f"Batch size: {len(paths)}")
    print(f"Features shape: {feats.shape}")
    print(f"Feature range: [{feats.min():.3f}, {feats.max():.3f}]")
    break

Batch size: 4
Features shape: (4, 2048)
Feature range: [0.000, 6.517]


In [5]:
print("Extracting deep features from all images...")
print("Using ResNet50 (frozen). Expected: 8-20 minutes on CPU.")
print()

records = []
batch_size = 32

for split in ['Training', 'Testing']:
    for c in CLASSES:
        in_dir = PROC / split / c
        files = sorted(in_dir.glob('*.png'))
        
        for paths, feats in tqdm(
            list(extract_deep_batch(files, batch_size=batch_size)),
            desc=f'{split}/{c}'
        ):
            for path, feat_vec in zip(paths, feats):
                row = {
                    'image_id': path.stem,
                    'split': split,
                    'class': c,
                }
                # Add 2048 deep features as deep_0, deep_1, ..., deep_2047
                for i, v in enumerate(feat_vec):
                    row[f'deep_{i}'] = float(v)
                records.append(row)

print(f"\nDone! Extracted features for {len(records)} images.")

Extracting deep features from all images...
Using ResNet50 (frozen). Expected: 8-20 minutes on CPU.



Training/glioma:   0%|          | 0/44 [00:00<?, ?it/s]

Training/meningioma:   0%|          | 0/44 [00:00<?, ?it/s]

Training/pituitary:   0%|          | 0/44 [00:00<?, ?it/s]

Training/notumor:   0%|          | 0/44 [00:00<?, ?it/s]

Testing/glioma:   0%|          | 0/13 [00:00<?, ?it/s]

Testing/meningioma:   0%|          | 0/13 [00:00<?, ?it/s]

Testing/pituitary:   0%|          | 0/13 [00:00<?, ?it/s]

Testing/notumor:   0%|          | 0/13 [00:00<?, ?it/s]


Done! Extracted features for 7200 images.


In [6]:
deep_df = pd.DataFrame(records)
print(f"DataFrame shape: {deep_df.shape}")
print(f"Expected: ({len(records)}, 2051) — 2048 features + 3 metadata")

# Save
out_path = FEAT / 'deep.csv'
deep_df.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path.resolve()}")
print(f"File size: {out_path.stat().st_size / 1024 / 1024:.1f} MB")

DataFrame shape: (7200, 2051)
Expected: (7200, 2051) — 2048 features + 3 metadata

Saved to: C:\Users\HP\Desktop\MU\bmi\brain_tumor_fusion\features\deep.csv
File size: 181.5 MB


In [7]:
# Check for NaN/Inf
nans = deep_df.iloc[:, 3:].isna().sum().sum()
infs = (~np.isfinite(deep_df.iloc[:, 3:].values)).sum()
print(f"NaN values: {nans}")
print(f"Inf values: {infs}")

# Distribution of one feature across classes
print("\nMean of deep_0 across classes:")
print(deep_df.groupby('class')['deep_0'].agg(['mean', 'std']))

NaN values: 0
Inf values: 0

Mean of deep_0 across classes:
                mean       std
class                         
glioma      0.066966  0.156964
meningioma  0.043525  0.123755
notumor     0.033716  0.095875
pituitary   0.016019  0.038870


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

feat_cols = [c for c in deep_df.columns if c.startswith('deep_')]

train = deep_df[deep_df['split'] == 'Training']
test  = deep_df[deep_df['split'] == 'Testing']

X_train = train[feat_cols].values
X_test  = test[feat_cols].values

le = LabelEncoder()
y_train = le.fit_transform(train['class'])
y_test  = le.transform(test['class'])

sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc  = sc.transform(X_test)

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
pred = rf.predict(X_test_sc)

print(f"Deep-only baseline accuracy: {accuracy_score(y_test, pred):.4f}")
print()
print(classification_report(y_test, pred, target_names=le.classes_))

Deep-only baseline accuracy: 0.8569

              precision    recall  f1-score   support

      glioma       0.91      0.62      0.74       400
  meningioma       0.73      0.87      0.79       400
     notumor       0.91      1.00      0.95       400
   pituitary       0.91      0.94      0.92       400

    accuracy                           0.86      1600
   macro avg       0.87      0.86      0.85      1600
weighted avg       0.87      0.86      0.85      1600



In [9]:
#verification with logistic regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=2000, n_jobs=-1, C=0.1)  # C=0.1 = mild regularization
lr.fit(X_train_sc, y_train)
pred_lr = lr.predict(X_test_sc)

print(f"Deep features + Logistic Regression: {accuracy_score(y_test, pred_lr):.4f}")
print()
print(classification_report(y_test, pred_lr, target_names=le.classes_))

Deep features + Logistic Regression: 0.9000

              precision    recall  f1-score   support

      glioma       0.90      0.74      0.81       400
  meningioma       0.83      0.90      0.86       400
     notumor       0.92      1.00      0.96       400
   pituitary       0.95      0.96      0.96       400

    accuracy                           0.90      1600
   macro avg       0.90      0.90      0.90      1600
weighted avg       0.90      0.90      0.90      1600



In [10]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

rf_pca = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_pca.fit(X_train_pca, y_train)
pred_pca = rf_pca.predict(X_test_pca)

print(f"\nDeep features (PCA-100) + Random Forest: {accuracy_score(y_test, pred_pca):.4f}")
print(classification_report(y_test, pred_pca, target_names=le.classes_))

PCA explained variance: 0.511

Deep features (PCA-100) + Random Forest: 0.8644
              precision    recall  f1-score   support

      glioma       0.90      0.65      0.75       400
  meningioma       0.75      0.86      0.80       400
     notumor       0.92      1.00      0.96       400
   pituitary       0.91      0.94      0.93       400

    accuracy                           0.86      1600
   macro avg       0.87      0.86      0.86      1600
weighted avg       0.87      0.86      0.86      1600

